# BiasAuditFW - local MLLM-as-a-Judge validation

This optional Colab/T4 workflow evaluates two open multimodal model families.
It runs a semantic pilot before any full batch and never modifies the canonical
`audit_images.csv` or `audit_faces.csv`. Human participant records must not be
uploaded to this notebook.

In [ ]:
from pathlib import Path

REPOSITORY_ARCHIVE = Path(
    "/content/drive/MyDrive/BiasAuditFW/snapshots/"
    "BiasAuditFW_mllm_validation_snapshot_v10.zip"
)
REPOSITORY_DIR = Path("/content/bias-audit-framework")

DATASET_ID = "original-64"
INPUT_ROOT = Path("/content/drive/MyDrive/Estudo UNAL/Images Generated by IA")
AUDIT_IMAGES_CSV = Path(
    "/content/drive/MyDrive/Estudo UNAL/"
    "BiasAuditFW outputs schema 2/full/audit_images.csv"
)
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Estudo UNAL/"
    "BiasAuditFW outputs schema 2/mllm-local"
) / DATASET_ID

RUN_QWEN3_PILOT = True
RUN_GEMMA_PILOT = True
RUN_FULL_QWEN3 = False
RUN_FULL_GEMMA = False
FULL_SAMPLE_CSV = None  # Path to this corpus' frozen human-validation sample.

QWEN3_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
GEMMA_MODEL_ID = "google/gemma-3-4b-it"
QWEN3_MODEL_REVISION = "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"
GEMMA_MODEL_REVISION = "093f9f388b31de276ce2de164bdc2081324b9767"


## 1. Mount Drive and install the frozen snapshot

The snapshot hash identifies the exact code used in the experiment.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import hashlib
import shutil
import subprocess
import sys

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
if not REPOSITORY_ARCHIVE.is_file():
    raise FileNotFoundError(REPOSITORY_ARCHIVE)

shutil.unpack_archive(str(REPOSITORY_ARCHIVE), str(REPOSITORY_DIR))
project_files = list(REPOSITORY_DIR.rglob("pyproject.toml"))
if len(project_files) != 1:
    raise RuntimeError(f"Expected one pyproject.toml, found {len(project_files)}")
REPOSITORY_DIR = project_files[0].parent
REVISION = "archive-sha256:" + hashlib.sha256(
    REPOSITORY_ARCHIVE.read_bytes()
).hexdigest()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--editable",
        f"{REPOSITORY_DIR}[judges,neural,test]",
    ],
    check=True,
)
print("Repository revision:", REVISION)

## 2. Load the Hugging Face secret and validate inputs

Create a Colab secret named `HF_TOKEN`, grant notebook access, and accept the
Gemma license on Hugging Face. The token is read without being printed.

In [ ]:
import os

import pandas as pd
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is required for the gated Gemma model")
os.environ["HF_TOKEN"] = hf_token

images = pd.read_csv(AUDIT_IMAGES_CSV)
required = {
    "dataset_id",
    "imagem_id",
    "content_sha256",
    "caminho_relativo",
    "pair_id",
    "tipo_prompt",
}
if missing := required - set(images):
    raise ValueError(f"Missing columns: {sorted(missing)}")
if images["imagem_id"].duplicated().any():
    raise ValueError("Duplicate imagem_id values")
missing_files = [
    relative
    for relative in images["caminho_relativo"]
    if not (INPUT_ROOT / relative).is_file()
]
if missing_files:
    raise FileNotFoundError(missing_files[:3])

print(f"Validated {len(images)} images and {images['pair_id'].nunique()} pairs")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_judges.py",
        "tests/test_judge_quality.py",
        "tests/test_human_mllm_statistics.py",
        "tests/test_sampling.py",
    ],
    cwd=REPOSITORY_DIR,
    check=True,
)

## 3. Run independent semantic pilots

Each model evaluates the same four complete pairs in both A/B orders. A failed
gate is preserved as a diagnostic result and does not prevent the other model
from being tested.

In [ ]:
import json

PILOT_SCRIPT = REPOSITORY_DIR / "scripts/run_mllm_semantic_pilot.py"
COMMON = [
    "--images-csv",
    str(AUDIT_IMAGES_CSV),
    "--input-root",
    str(INPUT_ROOT),
]

PILOTS = {
    "qwen3": {
        "enabled": RUN_QWEN3_PILOT,
        "directory": OUTPUT_DIR / "pilot-qwen3vl8b-v1",
        "arguments": [
            "--qwen3-model-id",
            QWEN3_MODEL_ID,
            "--qwen3-revision",
            QWEN3_MODEL_REVISION,
            "--qwen3-quantization",
            "4bit-nf4",
        ],
    },
    "gemma": {
        "enabled": RUN_GEMMA_PILOT,
        "directory": OUTPUT_DIR / "pilot-gemma3-4b-v1",
        "arguments": [
            "--gemma-model-id",
            GEMMA_MODEL_ID,
            "--gemma-revision",
            GEMMA_MODEL_REVISION,
            "--gemma-quantization",
            "4bit-nf4",
        ],
    },
}


def run_pilot(provider_name, config):
    if not config["enabled"]:
        return None
    directory = config["directory"]
    command = [
        sys.executable,
        str(PILOT_SCRIPT),
        *COMMON,
        "--output-dir",
        str(directory),
        "--provider",
        provider_name,
        *config["arguments"],
    ]
    completed = subprocess.run(command, cwd=REPOSITORY_DIR, check=False)
    report_path = directory / "semantic_pilot_report.json"
    if not report_path.is_file():
        raise RuntimeError(
            f"{provider_name} failed before producing a gate report "
            f"(exit code {completed.returncode})"
        )
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(provider_name, "passed=" + str(report["passed"]))
    return report


PILOT_REPORTS = {
    name: run_pilot(name, config)
    for name, config in PILOTS.items()
    if config["enabled"]
}

## 4. Optional full execution after gate approval

Keep both full-run flags `False` until the pilot reports have been reviewed.
The full execution is limited to a frozen corpus-specific sample CSV. Run this
notebook once for each corpus after the private 48-position form map has been
split into its corresponding corpus samples.

In [ ]:
SCRIPT = REPOSITORY_DIR / "scripts/run_mllm_validation.py"


def run_full(provider_name, enabled, pilot_directory, provider_arguments):
    if not enabled:
        return
    if FULL_SAMPLE_CSV is None:
        raise RuntimeError(
            "Set FULL_SAMPLE_CSV to this corpus' frozen human-validation "
            "sample before enabling a full run"
        )
    sample_path = Path(FULL_SAMPLE_CSV)
    if not sample_path.is_file():
        raise FileNotFoundError(sample_path)
    report_path = pilot_directory / "semantic_pilot_report.json"
    if not report_path.is_file():
        raise RuntimeError(f"Missing pilot report for {provider_name}")
    report = json.loads(report_path.read_text(encoding="utf-8"))
    if not report["passed"]:
        raise RuntimeError(f"A passing {provider_name} pilot is required")
    subprocess.run(
        [
            sys.executable,
            str(SCRIPT),
            *COMMON,
            "--output-dir",
            str(OUTPUT_DIR / f"full-{provider_name}-v1"),
            "--providers",
            provider_name,
            "--mode",
            "both",
            "--sample-csv",
            str(sample_path),
            *provider_arguments,
        ],
        cwd=REPOSITORY_DIR,
        check=True,
    )


run_full(
    "qwen3",
    RUN_FULL_QWEN3,
    PILOTS["qwen3"]["directory"],
    PILOTS["qwen3"]["arguments"],
)
run_full(
    "gemma",
    RUN_FULL_GEMMA,
    PILOTS["gemma"]["directory"],
    PILOTS["gemma"]["arguments"],
)

## 5. Review artifacts

`passed=true` is only an engineering degeneration screen. It does not establish
accuracy; academic validity still depends on comparison with the aggregated
UNESP perceptual reference.

In [ ]:
for provider_name, config in PILOTS.items():
    directory = config["directory"]
    print("
", provider_name, directory)
    if not directory.is_dir():
        print("not executed")
        continue
    for path in sorted(directory.glob("*")):
        print(path.name, path.stat().st_size, "bytes")
    pointwise_path = directory / "judge_pointwise.csv"
    if pointwise_path.exists():
        display(pd.read_csv(pointwise_path))
    report_path = directory / "semantic_pilot_report.json"
    if report_path.exists():
        display(json.loads(report_path.read_text(encoding="utf-8")))